# **Datos estructurados**

>**Importante**   
Tenemos que tener completado el paso anterior "02-obtener-api-key.md"

In [ ]:
#Instalar dependencias
%pip install anthropic python-dotenv

In [3]:
#Guardar y cargar API KEY
#Para guardar la key ponemos un .env y dentro ANTHROPIC_API_KEY="NUESTRA KEY"
#Importante recordar poner el .env en el gitignore
from dotenv import load_dotenv
load_dotenv()

True

In [12]:
#Crear un cliente API
from anthropic import Anthropic

cliente = Anthropic()
model = "claude-sonnet-5-5"

---

## Historial

Es importante saber que Claude no almacena información sobre los mensajes e historial que se ha ido teniendo con la misma.   
Por este motivo trabajaremos en un almacén de mensajes que pasarle para que Claude pueda tener conocimeinto de todo.

In [14]:

# 1. En esta función solicitamos la lista de mensajes totales y un texto para introducir un nuevo mensaje
# definimos el rol y contexto de forma rápida y añadimos con un append el mensaje nuevo a la lista total.
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

#2. Hacemos lo mismo con el mensaje de respuesta, como rol tendrá assistant
def add_assistant_message(messages, text):
    assistante_message = {"role": "assistant", "content": text}
    messages.append(assistante_message)

#3. Funcion automatizada en la que solicitamos la lista de mensajes y retornamos la respuesta habiéndo pasado el historial completo
def chat(messages, system):
    message = cliente.messages.create(
        system=system,
        model=model,
        max_tokens=100,
        messages=messages
    )
    for block in message.content:
        if block.type == "text":
            return block.text
    if message.stop_reason == "max_tokens":
        print("Límite de tokens alcanzado")
        

In [15]:
#Lista vacía de mensajes
messages = []

system = """Eres un ayudante de python, tienes que responder de la forma más sencilla posible 
            y sin comentarios en el texto lo que solicite el usuario"""

#Añadimos la pregunta inicial del user a nuestra lista
add_user_message(messages, "Dame una función compleja para hacer una conexión a una base de datos postgres + python")

#Obtenemos la respuesta del modelo
answer = chat(messages, system)

#Almacenamos la respuesta
add_assistant_message(messages, answer)

#Para testear nuestro historial vamos a solicitarle sin contexto otro animal
add_user_message(messages, "Dame otra diferente")

final_answer = chat(messages, system)

#Y volvemos a guardarlo
add_assistant_message(messages, final_answer)

Límite de tokens alcanzado


BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.1.content: Input should be a valid array'}, 'request_id': 'req_011CgYrwFrtKfHmaXdczDAse'}

In [11]:
for msg in messages:
    print(msg)

{'role': 'user', 'content': 'Dame una función para hacer una calculadora en python'}
{'role': 'assistant', 'content': '```python\ndef calculadora(a, b, operador):\n    if operador == \'+\':\n        return a + b\n    elif operador == \'-\':\n        return a - b\n    elif operador == \'*\':\n        return a * b\n    elif operador == \'/\':\n        if b == 0:\n            return "Error: división entre cero"\n        return a / b'}
{'role': 'user', 'content': 'Dame otra diferente'}
{'role': 'assistant', 'content': '```python\ndef calculadora(a, b, operador):\n    operaciones = {\n        \'+\': lambda x, y: x + y,\n        \'-\': lambda x, y: x - y,\n        \'*\': lambda x, y: x * y,\n        \'/\': lambda x, y: x / y if y != 0 else "Error: división ent'}
